In [1]:
import os, random, numpy as np, tensorflow as tf, pandas as pd

os.environ["PYTHONHASHSEED"] = "0"
random.seed(0)
np.random.seed(0)
tf.random.set_seed(0)


In [2]:
base_dir = "dataset/chest_xray/chest_xray"   # change if needed
trainval_dir = os.path.join(base_dir, "train")  # we'll also include val manually below
val_dir_kaggle = os.path.join(base_dir, "val")
test_dir = os.path.join(base_dir, "test")

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen_mnv3 = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=5,
    width_shift_range=0.02,
    height_shift_range=0.02,
    zoom_range=0.05,
    brightness_range=(0.9, 1.1),
    horizontal_flip=False
)

val_datagen_mnv3 = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

test_datagen_mnv3 = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen_mnv3 = train_datagen_mnv3.flow_from_directory(
    trainval_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training",
    shuffle=True
)

val_gen_mnv3 = val_datagen_mnv3.flow_from_directory(
    trainval_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    shuffle=False
)

test_gen_mnv3 = test_datagen_mnv3.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)


Found 4173 images belonging to 2 classes.
Found 1043 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [4]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV3Large

def build_mobilenetv3_large(dropout=0.3):
    base = MobileNetV3Large(
        weights="imagenet",
        include_top=False,
        input_shape=IMG_SIZE + (3,)
    )
    base.trainable = False  # Stage 1

    inputs = layers.Input(shape=IMG_SIZE + (3,))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auroc"),
            tf.keras.metrics.AUC(name="auprc", curve="PR"),
        ],
    )
    return model

mnv3 = build_mobilenetv3_large()
mnv3.summary()


2026-04-27 21:18:58.376451: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-04-27 21:18:58.376500: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-04-27 21:18:58.376512: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
2026-04-27 21:18:58.376560: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-27 21:18:58.376573: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Large (Functional)   │ (None, 7, 7, 960)      │     2,996,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 960)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 960)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           961 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,997,313 (11.43 MB)

 Trainable params: 961 (3.75 KB)

 Non-trainable params: 2,996,352 (11.43 MB)

In [5]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

callbacks_mnv3 = [
    ModelCheckpoint(
        "best_mnv3.keras",
        monitor="val_auprc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_auprc",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_auprc",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]


## Stage 1 trainig ( head only)

In [6]:
hist1_mnv3 = mnv3.fit(
    train_gen_mnv3,
    validation_data=val_gen_mnv3,
    epochs=10,
    callbacks=callbacks_mnv3
)


Epoch 1/10


2026-04-27 21:19:15.451888: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 305ms/step - accuracy: 0.7658 - auprc: 0.9059 - auroc: 0.7840 - loss: 0.4930
Epoch 1: val_auprc improved from None to 0.99291, saving model to best_mnv3.keras

Epoch 1: finished saving model to best_mnv3.keras
131/131 ━━━━━━━━━━━━━━━━━━━━ 56s 383ms/step - accuracy: 0.8421 - auprc: 0.9561 - auroc: 0.8898 - loss: 0.3563 - val_accuracy: 0.9223 - val_auprc: 0.9929 - val_auroc: 0.9787 - val_loss: 0.1997 - learning_rate: 0.0010
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 279ms/step - accuracy: 0.9237 - auprc: 0.9879 - auroc: 0.9643 - loss: 0.2117
Epoch 2: val_auprc improved from 0.99291 to 0.99618, saving model to best_mnv3.keras

Epoch 2: finished saving model to best_mnv3.keras
131/131 ━━━━━━━━━━━━━━━━━━━━ 42s 319ms/step - accuracy: 0.9231 - auprc: 0.9882 - auroc: 0.9691 - loss: 0.2002 - val_accuracy: 0.9482 - val_auprc: 0.9962 - val_auroc: 0.9887 - val_loss: 0.1426 - learning_rate: 0.0010
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 282ms/step - accuracy: 0.94

## Stage 2 fine-tuning

In [7]:
mnv3.load_weights("best_mnv3.keras")

base = mnv3.layers[1]
base.trainable = True

for layer in base.layers[:-20]:
    layer.trainable = False

mnv3.compile(
    optimizer=tf.keras.optimizers.Adam(5e-6),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auroc"),
        tf.keras.metrics.AUC(name="auprc", curve="PR"),
    ],
)

hist2_mnv3 = mnv3.fit(
    train_gen_mnv3,
    validation_data=val_gen_mnv3,
    epochs=10,
    callbacks=callbacks_mnv3
)


Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 292ms/step - accuracy: 0.9311 - auprc: 0.9942 - auroc: 0.9833 - loss: 0.1607
Epoch 1: val_auprc improved from 0.99876 to 0.99880, saving model to best_mnv3.keras

Epoch 1: finished saving model to best_mnv3.keras
131/131 ━━━━━━━━━━━━━━━━━━━━ 56s 368ms/step - accuracy: 0.9331 - auprc: 0.9933 - auroc: 0.9809 - loss: 0.1656 - val_accuracy: 0.9588 - val_auprc: 0.9988 - val_auroc: 0.9965 - val_loss: 0.0947 - learning_rate: 5.0000e-06
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 288ms/step - accuracy: 0.9435 - auprc: 0.9952 - auroc: 0.9859 - loss: 0.1395
Epoch 2: val_auprc improved from 0.99880 to 0.99884, saving model to best_mnv3.keras

Epoch 2: finished saving model to best_mnv3.keras

Epoch 2: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-06.
131/131 ━━━━━━━━━━━━━━━━━━━━ 43s 330ms/step - accuracy: 0.9425 - auprc: 0.9951 - auroc: 0.9852 - loss: 0.1421 - val_accuracy: 0.9597 - val_auprc: 0.9988 - val_auroc: 0.9966 - val_loss: 0.0969

In [10]:
from utils import eval_thresholds


In [11]:
thresholds = np.linspace(0.2, 0.9, 71)
val_table_mnv3 = eval_thresholds(mnv3, val_gen_mnv3, thresholds)
val_table_mnv3


,Threshold,Accuracy,Macro F1,Weighted F1,Specificity,Sensitivity,FN,FP
0,0.20,0.976,0.969,0.976,0.955,0.983,13,12
1,0.21,0.975,0.967,0.975,0.955,0.982,14,12
2,0.22,0.975,0.967,0.975,0.955,0.982,14,12
3,0.23,0.975,0.967,0.975,0.955,0.982,14,12
4,0.24,0.973,0.965,0.973,0.955,0.979,16,12
...,...,...,...,...,...,...,...,...
66,0.86,0.909,0.892,0.913,0.996,0.879,94,1
67,0.87,0.907,0.890,0.911,0.996,0.876,96,1
68,0.88,0.903,0.886,0.907,0.996,0.871,100,1
69,0.89,0.897,0.880,0.902,0.996,0.863,106,1


In [12]:
TARGET_SENS = 0.96

best_scr_mnv3 = (
    val_table_mnv3[val_table_mnv3["Sensitivity"] >= TARGET_SENS]
    .sort_values("Specificity", ascending=False)
    .iloc[0]
)

best_scr_mnv3


Threshold       0.320
Accuracy        0.975
Macro F1        0.968
Weighted F1     0.975
Specificity     0.985
Sensitivity     0.972
FN             22.000
FP              4.000
Name: 12, dtype: float64

In [13]:
from utils import evaluate


In [14]:
best_t_mnv3 = float(best_scr_mnv3["Threshold"])

mnv3_screen_res = evaluate(
    mnv3,
    test_gen_mnv3,
    name="MobileNetV3Large",
    threshold=best_t_mnv3
)

mnv3_screen_res["Mode"] = "Screening"
mnv3_screen_res["Params"] = mnv3.count_params()

mnv3_screen_res



=== MobileNetV3Large @ threshold 0.320 ===
[[140  94]
 [  4 386]]
              precision    recall  f1-score   support

      NORMAL       0.97      0.60      0.74       234
   PNEUMONIA       0.80      0.99      0.89       390

    accuracy                           0.84       624
   macro avg       0.89      0.79      0.81       624
weighted avg       0.87      0.84      0.83       624

AUROC: 0.9685 | AUPRC: 0.9809
Macro F1: 0.8140 | Weighted F1: 0.8324
Sensitivity: 0.9897 | Specificity: 0.5983 | Precision: 0.8042


{'Model': 'MobileNetV3Large',
 'Flip': None,
 'Class Weights': 'None',
 'Threshold': 0.32,
 'AUROC': 0.9685075608152531,
 'AUPRC': 0.9809410784047264,
 'Accuracy': 0.842948717948718,
 'Macro F1': 0.8140485312899106,
 'Weighted F1': 0.832375478927203,
 'Sensitivity': 0.9897435897435898,
 'Specificity': 0.5982905982905983,
 'Precision': 0.8041666666666667,
 'TN': 140,
 'FP': 94,
 'FN': 4,
 'TP': 386,
 'Mode': 'Screening',
 'Params': 2997313}

In [21]:
mnv3_screen_res = evaluate(
    mnv3,
    test_gen_mnv3,
    name="MobileNetV3Large",
    threshold=0.50
)


=== MobileNetV3Large @ threshold 0.500 ===
[[160  74]
 [  8 382]]
              precision    recall  f1-score   support

      NORMAL       0.95      0.68      0.80       234
   PNEUMONIA       0.84      0.98      0.90       390

    accuracy                           0.87       624
   macro avg       0.90      0.83      0.85       624
weighted avg       0.88      0.87      0.86       624

AUROC: 0.9685 | AUPRC: 0.9809
Macro F1: 0.8495 | Weighted F1: 0.8629
Sensitivity: 0.9795 | Specificity: 0.6838 | Precision: 0.8377


In [15]:
import pandas as pd
from pathlib import Path

out_dir = Path("results")
out_dir.mkdir(exist_ok=True)

pd.DataFrame([mnv3_screen_res]).to_csv(
    out_dir / "mobilenetv3_results.csv",
    index=False
)
